# Platt Lorrain Fine-Tuning with QLoRA

This notebook fine-tunes Ministral-3B to speak Platt Lorrain using QLoRA.

**Requirements:**
- Google Colab with T4 GPU (free tier works!)
- Upload `platt_chat_train.jsonl` to Colab

**Steps:**
1. Install dependencies
2. Load model with 4-bit quantization
3. Fine-tune with LoRA
4. Export adapter for Hugging Face

In [1]:
# Install Unsloth (2x faster than standard training)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!rm -rf /content/unsloth_compiled_cache/

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-pe_fukvo/unsloth_059132ed97004468b751d1ce9f3bd395
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-pe_fukvo/unsloth_059132ed97004468b751d1ce9f3bd395
  Resolved https://github.com/unslothai/unsloth.git to commit 997f1a7ce5fb7a0319c2b8abe0e7af02e2160efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 116.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.

In [2]:
from unsloth import FastLanguageModel
import torch

# Load Ministral-3B (multimodal, but we fine-tune text only)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Mistral-7B-Instruct-v0.3",
    max_seq_length=2048,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

print("Model loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Model loaded!


In [3]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Freeze vision tower — Ministral-3B is multimodal but we only need text fine-tuning
for name, param in model.named_parameters():
    if "vision_tower" in name:
        param.requires_grad = False

print("LoRA adapters added, vision tower frozen!")

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapters added, vision tower frozen!


In [4]:
# Download training data from GitHub
!wget -q https://raw.githubusercontent.com/yannklein/platt-bot/refs/heads/main/training/platt_chat_train.jsonl
import os
dataset_path = '/content/platt_chat_train.jsonl'
assert os.path.exists(dataset_path), "Download failed — check the file is committed and pushed to GitHub"
print(f"File ready: {dataset_path}")

File ready: /content/platt_chat_train.jsonl


In [5]:
from datasets import load_dataset

# Load training data
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"Loaded {len(dataset)} training examples")
print("\nSample:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 892 training examples

Sample:
{'messages': [{'role': 'system', 'content': 'Du bist ein freundlicher Assistent, der Platt Lorrain (Francique rhénan lorrain) spricht.\nDu antwortest IMMER auf Platt, egal in welcher Sprache die Frage gestellt wird.\nPlatt ist ein deutscher Dialekt aus Lothringen, nahe am Hochdeutschen aber mit eigenen Regeln:\n- "ich" bleibt "ich", aber Endungen werden weicher\n- "das" wird zu "dat"\n- "was" wird zu "wat"\n- Viele Wörter enden auf -e statt -en\n- Verkleinerungen enden auf -che (wie "Kanéngche")\nSei natürlich, freundlich und hilfsbereit - immer auf Platt!'}, {'role': 'user', 'content': 'Wie sagt man auf Platt: Letztes Jahr sind wir nach London gegangen.'}, {'role': 'assistant', 'content': 'Lèts Johr sinn mir noh London gàng.'}]}


In [6]:
# Format for Mistral Instruct
def format_chat(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print("\nFormatted sample:")
print(dataset[0]["text"][:500])

Map:   0%|          | 0/892 [00:00<?, ? examples/s]


Formatted sample:
<s>[INST] Wie sagt man auf Platt: Letztes Jahr sind wir nach London gegangen.[/INST] Lèts Johr sinn mir noh London gàng.</s>


In [8]:
# Format and tokenize dataset
from transformers import DataCollatorForLanguageModeling

def format_and_tokenize(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    result = tokenizer(text=text, truncation=True, max_length=2048, padding=False)
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(format_and_tokenize, remove_columns=dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f"Tokenized {len(tokenized_dataset)} examples")

Map:   0%|          | 0/892 [00:00<?, ? examples/s]

Tokenized 892 examples


In [9]:
from transformers import Trainer, TrainingArguments

# Use standard transformers Trainer — bypasses unsloth's compiled loop
# which doesn't support Ministral-3B's multimodal architecture
trainer = Trainer(
    model=model,
    # tokenizer=tokenizer, # Removed redundant tokenizer argument
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Trainer ready! Starting training...")

Trainer ready! Starting training...


In [11]:
# Test the model
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Du bist ein freundlicher Assistent, der Platt Lorrain spricht. Antworte immer auf Platt."},
    {"role": "user", "content": "Wie geht es dir heute?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=64,
    use_cache=True,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

Du bist ein freundlicher Assistent, der Platt Lorrain spricht. Antworte immer auf Platt.

Wie geht es dir heute? Mëscht de Schääf grouss, wéi dir? (How are you doing today, you?)


In [12]:
# Save LoRA adapter for Together AI
model.save_pretrained("platt-lorrain-lora")
tokenizer.save_pretrained("platt-lorrain-lora")

print("Adapter saved to platt-lorrain-lora/")

Adapter saved to platt-lorrain-lora/


In [ ]:
from huggingface_hub import HfApi

HfApi().upload_folder(
    folder_path="platt-lorrain-lora",
    repo_id="yannklein/platt-bot",
    repo_type="model",
    token="YOUR_HF_TOKEN_HERE",  # paste token, do NOT commit this
)

print("✅ Adapter pushed to yannklein/platt-bot")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  28%|##8       | 47.6MB /  168MB            

  ...rrain-lora/tokenizer.json: 100%|##########| 3.67MB / 3.67MB            

✅ Adapter pushed to yannklein/platt-bot


## Optional: Push to Hugging Face Hub

If you want to host on Hugging Face instead of Together AI:

In [14]:
# Optional: Push to Hugging Face Hub
# from huggingface_hub import login
# login()  # Enter your HF token
#
# model.push_to_hub("your-username/platt-lorrain-lora")
# tokenizer.push_to_hub("your-username/platt-lorrain-lora")